# Step 3: Exploring the downloaded financial data

Before we can calculate any scores (Altman Z'', Piotroski F-score, Beneish M-score), we need to know
**what data we actually have**. yfinance downloads whatever line items a company reports, and it does
not standardise every company to the exact same set of names. So this notebook:

1. Loads all 60 raw CSVs (20 companies x 3 statement types: balance sheet, income statement, cash flow)
2. Lists every line item name that appears anywhere, and how many companies have it
3. Checks whether a specific list of fields we will need later exists for each company, for the last 3 fiscal years
4. Saves a summary table so we can see the gaps at a glance

This notebook does not calculate anything or fix any gaps. It only reports what is there.

In [1]:
import pandas as pd
from pathlib import Path

# Project folders. Path("..") means "one level up from notebooks/", i.e. the project root.
PROJECT_ROOT = Path("..")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
COMPANIES_FILE = PROJECT_ROOT / "data" / "companies.csv"

# The three statement types our download script produced one CSV for, per company.
STATEMENT_TYPES = ["balance_sheet", "income_statement", "cash_flow"]

companies = pd.read_csv(COMPANIES_FILE)
companies.head(20)

,ticker,company_name,industry
0,ABBOTINDIA.NS,Abbott India Ltd.,Healthcare
1,AJANTPHARM.NS,Ajanta Pharmaceuticals Ltd.,Healthcare
2,ALKEM.NS,Alkem Laboratories Ltd.,Healthcare
3,AUROPHARMA.NS,Aurobindo Pharma Ltd.,Healthcare
4,BIOCON.NS,Biocon Ltd.,Healthcare
5,CIPLA.NS,Cipla Ltd.,Healthcare
6,DIVISLAB.NS,Divi's Laboratories Ltd.,Healthcare
7,DRREDDY.NS,Dr. Reddy's Laboratories Ltd.,Healthcare
8,GLAND.NS,Gland Pharma Ltd.,Healthcare
9,GLENMARK.NS,Glenmark Pharmaceuticals Ltd.,Healthcare


## Part 1: Load all 60 CSVs

Each raw CSV has one row per line item (e.g. "Total Revenue", "Total Assets") and one column per
fiscal year end date. The columns are already ordered newest-first (e.g. 2026-03-31, 2025-03-31, ...).

We load every file into one big dictionary, `raw_data`, keyed by `(ticker, statement_type)`, so the
rest of the notebook can look up any company's any statement in one line.

In [2]:
def ticker_to_filename_prefix(ticker):
    # Our download script saved files like "ABBOTINDIA_NS_balance_sheet.csv".
    # The ticker in companies.csv is "ABBOTINDIA.NS", so we swap the dot for an underscore.
    return ticker.replace(".", "_")


def load_statement(ticker, statement_type):
    # Reads one CSV and returns it indexed by line_item, so df.loc["Total Assets"] works directly.
    prefix = ticker_to_filename_prefix(ticker)
    file_path = RAW_DATA_DIR / f"{prefix}_{statement_type}.csv"
    df = pd.read_csv(file_path, index_col="line_item")
    return df


raw_data = {}
missing_files = []

for ticker in companies["ticker"]:
    for statement_type in STATEMENT_TYPES:
        try:
            raw_data[(ticker, statement_type)] = load_statement(ticker, statement_type)
        except FileNotFoundError:
            missing_files.append((ticker, statement_type))

print(f"Loaded {len(raw_data)} of {len(companies) * len(STATEMENT_TYPES)} expected files.")
if missing_files:
    print("Missing files:", missing_files)

Loaded 60 of 60 expected files.


## Part 2: Every line item name, and how many companies report it

For each statement type, we count how many of the 20 companies have a row with that exact name
(regardless of whether the values in that row are filled in or blank for every year -- this is just
"does this line item appear in the file at all").

This is useful because Yahoo Finance does not force every company to report the same line items --
for example, a company with no long-term debt may simply not have a "Long Term Debt" row at all.

In [3]:
def line_item_coverage(statement_type):
    # Counts, for one statement type, how many of the 20 companies have each line item name.
    counts = {}
    for ticker in companies["ticker"]:
        df = raw_data.get((ticker, statement_type))
        if df is None:
            continue
        for line_item in df.index:
            counts[line_item] = counts.get(line_item, 0) + 1
    coverage = pd.DataFrame(
        {"line_item": list(counts.keys()), "companies_with_this_field": list(counts.values())}
    )
    coverage["out_of"] = len(companies)
    coverage = coverage.sort_values("companies_with_this_field", ascending=False).reset_index(drop=True)
    return coverage


coverage_tables = {statement_type: line_item_coverage(statement_type) for statement_type in STATEMENT_TYPES}

In [4]:
print("BALANCE SHEET line items:")
coverage_tables["balance_sheet"]

BALANCE SHEET line items:


,line_item,companies_with_this_field,out_of
0,Ordinary Shares Number,20,20
1,Share Issued,20,20
2,Tangible Book Value,20,20
3,Total Debt,20,20
4,Invested Capital,20,20
...,...,...,...
113,Receivables,1,20
114,Preferred Stock Equity,1,20
115,Preferred Stock,1,20
116,Preferred Securities Outside Stock Equity,1,20


In [5]:
print("INCOME STATEMENT line items:")
coverage_tables["income_statement"]

INCOME STATEMENT line items:


,line_item,companies_with_this_field,out_of
0,Tax Effect Of Unusual Items,20,20
1,Tax Rate For Calcs,20,20
2,Normalized EBITDA,20,20
3,Total Unusual Items,20,20
4,Total Unusual Items Excluding Goodwill,20,20
...,...,...,...
59,Other Income Expense,1,20
60,Earnings From Equity Interest,1,20
61,Gain On Sale Of Security,1,20
62,Depreciation Amortization Depletion Income Sta...,1,20


In [6]:
print("CASH FLOW line items:")
coverage_tables["cash_flow"]

CASH FLOW line items:


,line_item,companies_with_this_field,out_of
0,Free Cash Flow,20,20
1,Capital Expenditure,20,20
2,End Cash Position,20,20
3,Beginning Cash Position,20,20
4,Changes In Cash,20,20
...,...,...,...
79,Earnings Losses From Equity Investments,1,20
80,Other Cash Adjustment Inside Changein Cash,1,20
81,Net Investment Properties Purchase And Sale,1,20
82,Sale Of Investment Properties,1,20


## Part 3: Do we have the fields we will need for scoring, for the last 3 years?

The Altman Z'', Piotroski F-score and Beneish M-score formulas all need specific inputs. Below is what
each required field means in plain accounting terms, and which exact Yahoo Finance line item(s) we
matched it to (Yahoo does not always use the same name we'd expect, so we check a short list of likely
names for each field and use the first one that exists).

| Required field | What it means | Yahoo field name(s) checked, in order |
|---|---|---|
| Total Revenue | Sales / turnover for the year | `Total Revenue` |
| Gross Profit | Revenue minus cost of goods sold | `Gross Profit` |
| EBIT | Earnings before interest and tax (operating profit) | `EBIT` |
| Net Income | Profit after all expenses, interest and tax | `Net Income` |
| Interest Expense | Interest paid on borrowings during the year | `Interest Expense` |
| Selling General And Administration | Overhead costs: admin staff, marketing, offices etc. (not factory/production costs) | `Selling General And Administration` |
| Depreciation | Wear-and-tear cost of fixed assets spread over their life | `Depreciation And Amortization In Income Statement`, else `Depreciation And Amortization` (from cash flow statement) |
| Total Assets | Everything the company owns | `Total Assets` |
| Current Assets | Assets expected to turn into cash within a year (cash, receivables, inventory) | `Current Assets` |
| Current Liabilities | Debts due within a year | `Current Liabilities` |
| Retained Earnings | Cumulative profit kept in the business rather than paid out as dividends | `Retained Earnings` |
| Stockholders Equity | Book value belonging to shareholders (assets minus liabilities) | `Stockholders Equity` |
| Total Liabilities | Everything the company owes | `Total Liabilities Net Minority Interest` (yfinance has no plain "Total Liabilities" field; this is its equivalent) |
| Accounts Receivable | Money owed to the company by customers who bought on credit | `Accounts Receivable`, else `Receivables` |
| Inventory | Unsold stock (raw materials, work in progress, finished goods) | `Inventory` |
| Net PPE | Property, plant and equipment, after depreciation | `Net PPE` |
| Long Term Debt | Borrowings due in more than one year | `Long Term Debt` |
| Total Debt | All interest-bearing borrowings, short and long term combined | `Total Debt` |
| Ordinary Shares Number | Number of shares outstanding | `Ordinary Shares Number`, else `Share Issued` |
| Operating Cash Flow | Cash generated from normal business operations | `Operating Cash Flow` |

Note on Selling General And Administration: some companies report this as one combined line, others
may only report separate "General And Administrative Expense" and "Selling And Marketing Expense"
lines instead. We only check for the combined field here, as required. If a company doesn't have it,
it will show up as missing below rather than being estimated.

In [7]:
# Each required field maps to a list of (statement_type, exact_line_item_name) candidates,
# checked in order. The first candidate that exists in a company's file is the one we use for it.
REQUIRED_FIELDS = {
    "Total Revenue": [("income_statement", "Total Revenue")],
    "Gross Profit": [("income_statement", "Gross Profit")],
    "EBIT": [("income_statement", "EBIT")],
    "Net Income": [("income_statement", "Net Income")],
    "Interest Expense": [("income_statement", "Interest Expense")],
    "Selling General And Administration": [("income_statement", "Selling General And Administration")],
    "Depreciation": [
        ("income_statement", "Depreciation And Amortization In Income Statement"),
        ("cash_flow", "Depreciation And Amortization"),
    ],
    "Total Assets": [("balance_sheet", "Total Assets")],
    "Current Assets": [("balance_sheet", "Current Assets")],
    "Current Liabilities": [("balance_sheet", "Current Liabilities")],
    "Retained Earnings": [("balance_sheet", "Retained Earnings")],
    "Stockholders Equity": [("balance_sheet", "Stockholders Equity")],
    "Total Liabilities": [("balance_sheet", "Total Liabilities Net Minority Interest")],
    "Accounts Receivable": [
        ("balance_sheet", "Accounts Receivable"),
        ("balance_sheet", "Receivables"),
    ],
    "Inventory": [("balance_sheet", "Inventory")],
    "Net PPE": [("balance_sheet", "Net PPE")],
    "Long Term Debt": [("balance_sheet", "Long Term Debt")],
    "Total Debt": [("balance_sheet", "Total Debt")],
    "Ordinary Shares Number": [
        ("balance_sheet", "Ordinary Shares Number"),
        ("balance_sheet", "Share Issued"),
    ],
    "Operating Cash Flow": [("cash_flow", "Operating Cash Flow")],
}

NUM_YEARS_REQUIRED = 3

In [8]:
def latest_n_year_columns(df, n):
    # Columns are already ordered newest-first by the download script, so we just take the first n.
    return list(df.columns[:n])


def find_matched_field(ticker, candidates):
    # Tries each (statement_type, line_item_name) candidate in order.
    # Returns the statement_type, the exact name matched, and the row of values, or (None, None, None).
    for statement_type, line_item_name in candidates:
        df = raw_data.get((ticker, statement_type))
        if df is None:
            continue
        if line_item_name in df.index:
            year_columns = latest_n_year_columns(df, NUM_YEARS_REQUIRED)
            values = df.loc[line_item_name, year_columns]
            return statement_type, line_item_name, values
    return None, None, None


def field_status(values):
    # values is a pandas Series of up to 3 numbers (one per fiscal year), possibly containing NaN.
    if values is None:
        return "missing"
    years_present = values.notna().sum()
    if years_present == 0:
        return "missing"
    if years_present == NUM_YEARS_REQUIRED:
        return "complete"
    return "partial"

### Which exact name did we match, per company?

This table shows, for every company and required field, the exact line item name (and which
statement it came from) that was actually used. "NOT FOUND" means none of the candidate names
existed anywhere in that company's files.

In [9]:
matched_name_rows = []

for ticker in companies["ticker"]:
    row = {"ticker": ticker}
    for field_name, candidates in REQUIRED_FIELDS.items():
        statement_type, matched_name, values = find_matched_field(ticker, candidates)
        if matched_name is None:
            row[field_name] = "NOT FOUND"
        else:
            row[field_name] = f"{matched_name} ({statement_type})"
    matched_name_rows.append(row)

matched_names = pd.DataFrame(matched_name_rows).set_index("ticker")
matched_names

,Total Revenue,Gross Profit,EBIT,Net Income,Interest Expense,Selling General And Administration,Depreciation,Total Assets,Current Assets,Current Liabilities,Retained Earnings,Stockholders Equity,Total Liabilities,Accounts Receivable,Inventory,Net PPE,Long Term Debt,Total Debt,Ordinary Shares Number,Operating Cash Flow
ticker,,,,,,,,,,,,,,,,,,,,
ABBOTINDIA.NS,Total Revenue (income_statement),Gross Profit (income_statement),EBIT (income_statement),Net Income (income_statement),Interest Expense (income_statement),Selling General And Administration (income_sta...,Depreciation And Amortization In Income Statem...,Total Assets (balance_sheet),Current Assets (balance_sheet),Current Liabilities (balance_sheet),Retained Earnings (balance_sheet),Stockholders Equity (balance_sheet),Total Liabilities Net Minority Interest (balan...,Accounts Receivable (balance_sheet),Inventory (balance_sheet),Net PPE (balance_sheet),NOT FOUND,Total Debt (balance_sheet),Ordinary Shares Number (balance_sheet),Operating Cash Flow (cash_flow)
AJANTPHARM.NS,Total Revenue (income_statement),Gross Profit (income_statement),EBIT (income_statement),Net Income (income_statement),Interest Expense (income_statement),Selling General And Administration (income_sta...,Depreciation And Amortization In Income Statem...,Total Assets (balance_sheet),Current Assets (balance_sheet),Current Liabilities (balance_sheet),Retained Earnings (balance_sheet),Stockholders Equity (balance_sheet),Total Liabilities Net Minority Interest (balan...,Accounts Receivable (balance_sheet),Inventory (balance_sheet),Net PPE (balance_sheet),Long Term Debt (balance_sheet),Total Debt (balance_sheet),Ordinary Shares Number (balance_sheet),Operating Cash Flow (cash_flow)
ALKEM.NS,Total Revenue (income_statement),Gross Profit (income_statement),EBIT (income_statement),Net Income (income_statement),Interest Expense (income_statement),Selling General And Administration (income_sta...,Depreciation And Amortization In Income Statem...,Total Assets (balance_sheet),Current Assets (balance_sheet),Current Liabilities (balance_sheet),Retained Earnings (balance_sheet),Stockholders Equity (balance_sheet),Total Liabilities Net Minority Interest (balan...,Accounts Receivable (balance_sheet),Inventory (balance_sheet),Net PPE (balance_sheet),Long Term Debt (balance_sheet),Total Debt (balance_sheet),Ordinary Shares Number (balance_sheet),Operating Cash Flow (cash_flow)
AUROPHARMA.NS,Total Revenue (income_statement),Gross Profit (income_statement),EBIT (income_statement),Net Income (income_statement),Interest Expense (income_statement),Selling General And Administration (income_sta...,Depreciation And Amortization In Income Statem...,Total Assets (balance_sheet),Current Assets (balance_sheet),Current Liabilities (balance_sheet),Retained Earnings (balance_sheet),Stockholders Equity (balance_sheet),Total Liabilities Net Minority Interest (balan...,Accounts Receivable (balance_sheet),Inventory (balance_sheet),Net PPE (balance_sheet),Long Term Debt (balance_sheet),Total Debt (balance_sheet),Ordinary Shares Number (balance_sheet),Operating Cash Flow (cash_flow)
BIOCON.NS,Total Revenue (income_statement),Gross Profit (income_statement),EBIT (income_statement),Net Income (income_statement),Interest Expense (income_statement),Selling General And Administration (income_sta...,Depreciation And Amortization In Income Statem...,Total Assets (balance_sheet),Current Assets (balance_sheet),Current Liabilities (balance_sheet),Retained Earnings (balance_sheet),Stockholders Equity (balance_sheet),Total Liabilities Net Minority Interest (balan...,Accounts Receivable (balance_sheet),Inventory (balance_sheet),Net PPE (balance_sheet),Long Term Debt (balance_sheet),Total Debt (balance_sheet),Ordinary Shares Number (balance_sheet),Operating Cash Flow (cash_flow)
CIPLA.NS,Total Revenue (income_statement),Gross Profit (income_statement),EBIT (income_statement),Net Income (income_statement),Interest Expense (income_statement),Selling General And

### Companies x required fields: complete / partial / missing

- **complete**: the field exists and has a value in all 3 of the latest fiscal years
- **partial**: the field exists but is blank (NaN) for at least one of the 3 latest fiscal years
- **missing**: the field does not appear in the company's file at all, or is blank in all 3 years

In [10]:
status_rows = []

for ticker in companies["ticker"]:
    row = {"ticker": ticker}
    for field_name, candidates in REQUIRED_FIELDS.items():
        _, _, values = find_matched_field(ticker, candidates)
        row[field_name] = field_status(values)
    status_rows.append(row)

field_availability = pd.DataFrame(status_rows).set_index("ticker")
field_availability

,Total Revenue,Gross Profit,EBIT,Net Income,Interest Expense,Selling General And Administration,Depreciation,Total Assets,Current Assets,Current Liabilities,Retained Earnings,Stockholders Equity,Total Liabilities,Accounts Receivable,Inventory,Net PPE,Long Term Debt,Total Debt,Ordinary Shares Number,Operating Cash Flow
ticker,,,,,,,,,,,,,,,,,,,,
ABBOTINDIA.NS,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,missing,complete,complete,complete
AJANTPHARM.NS,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,partial,complete,complete,complete
ALKEM.NS,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete
AUROPHARMA.NS,complete,complete,complete,complete,complete,partial,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete
BIOCON.NS,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete
CIPLA.NS,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete
DIVISLAB.NS,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,missing,complete,complete,complete
DRREDDY.NS,complete,complete,complete,complete,complete,complete,partial,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete
GLAND.NS,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete,complete


In [11]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
output_path = PROCESSED_DATA_DIR / "field_availability.csv"
field_availability.to_csv(output_path)
print(f"Saved to {output_path}")

Saved to ..\data\processed\field_availability.csv


In [12]:
# Helper counts used in the summary below.
missing_count_by_field = (field_availability == "missing").sum().sort_values(ascending=False)
partial_count_by_field = (field_availability == "partial").sum()
gap_count_by_company = (field_availability != "complete").sum(axis=1).sort_values(ascending=False)

print("Missing count by field (most missing first):")
print(missing_count_by_field)
print()
print("Partial count by field:")
print(partial_count_by_field)
print()
print("Gap count by company (missing + partial, most gaps first):")
print(gap_count_by_company)

Missing count by field (most missing first):
Long Term Debt                        2
Total Revenue                         0
EBIT                                  0
Gross Profit                          0
Net Income                            0
Interest Expense                      0
Depreciation                          0
Selling General And Administration    0
Current Assets                        0
Current Liabilities                   0
Retained Earnings                     0
Total Assets                          0
Stockholders Equity                   0
Total Liabilities                     0
Inventory                             0
Accounts Receivable                   0
Net PPE                               0
Total Debt                            0
Ordinary Shares Number                0
Operating Cash Flow                   0
dtype: int64

Partial count by field:
Total Revenue                         0
Gross Profit                          0
EBIT                                 

## Summary

Out of 20 companies and 20 required fields (400 company-field combinations), the data is largely
clean: **14 of the 20 companies have every required field complete for all 3 years**, and 16 of
the 20 required fields are complete for every single company.

**Most-missing field:** `Long Term Debt` is the only field that is fully missing for any company --
it doesn't appear at all for **ABBOTINDIA.NS** and **DIVISLAB.NS**. This most likely means these two
companies simply have no long-term borrowings on their books, so yfinance has no row for it (worth
confirming against the actual balance sheet rather than assuming this).

**Fields with partial (not all 3 years) coverage**, one company each:
- `Long Term Debt` -- AJANTPHARM.NS
- `Selling General And Administration` -- AUROPHARMA.NS
- `Depreciation` -- DRREDDY.NS
- `Retained Earnings` -- TORNTPHARM.NS

**Companies with the most gaps:** no company has more than one gap. The 6 companies with any gap at
all are ABBOTINDIA.NS, AJANTPHARM.NS, AUROPHARMA.NS, DIVISLAB.NS, DRREDDY.NS and TORNTPHARM.NS, each
missing/partial on exactly one field. The other 14 companies are fully complete on all 20 fields.

No values were filled in or estimated anywhere in this notebook -- these gaps are exactly what's in
the raw downloaded data, and will need a decision (e.g. treat missing Long Term Debt as zero, drop
the field from scoring, or drop the company) before we calculate any scores in the next step.